# Machine Learning System for Credit Card Fraud Detection
Engineering project notebook. Run all cells top to bottom (Runtime > Run all). Training takes roughly 10-15 minutes on Colab.

In [ ]:
!pip -q install xgboost imbalanced-learn shap kagglehub
import os
for d in ['data','models','figures','results']: os.makedirs(d,exist_ok=True)

## 0. Download the dataset (free)
Source: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud . If kagglehub asks for login, download `creditcard.csv` manually from Kaggle and upload it to `data/` instead.

In [ ]:
import kagglehub, shutil
try:
    p=kagglehub.dataset_download('mlg-ulb/creditcardfraud'); shutil.copy(os.path.join(p,'creditcard.csv'),'data/creditcard.csv')
except Exception as e:
    print('Download failed, upload creditcard.csv to data/ manually:',e)

In [ ]:
import numpy as np
from sklearn.metrics import precision_score,recall_score,f1_score,average_precision_score,roc_auc_score,confusion_matrix,precision_recall_curve
def best_thr(y,p):
    pr,rc,th=precision_recall_curve(y,p)
    f=2*pr[:-1]*rc[:-1]/np.maximum(pr[:-1]+rc[:-1],1e-12)
    i=int(np.argmax(f));return float(th[i])
def metrics(y,p,thr=0.5):
    yh=(p>=thr).astype(int);tn,fp,fn,tp=confusion_matrix(y,yh).ravel()
    return dict(precision=precision_score(y,yh,zero_division=0),recall=recall_score(y,yh),f1=f1_score(y,yh),
        pr_auc=average_precision_score(y,p),roc_auc=roc_auc_score(y,p),thr=thr,tp=int(tp),fp=int(fp),fn=int(fn),tn=int(tn))


## 1. Data preparation
Remove duplicates, stratified 70/15/15 split, scale Time and Amount (fit on train only).

In [ ]:
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
df=pd.read_csv('data/creditcard.csv')
print('raw',df.shape,'dups',df.duplicated().sum(),'nulls',df.isna().sum().sum())
df=df.drop_duplicates().reset_index(drop=True)
print('clean',df.shape,'fraud',df.Class.sum(),f'{df.Class.mean()*100:.3f}%')
X=df.drop(columns='Class');y=df.Class
Xtr,Xtmp,ytr,ytmp=train_test_split(X,y,test_size=0.30,stratify=y,random_state=42)
Xva,Xte,yva,yte=train_test_split(Xtmp,ytmp,test_size=0.50,stratify=ytmp,random_state=42)
sc=RobustScaler().fit(Xtr[['Time','Amount']])
def tr(a):
    a=a.copy();a[['Time','Amount']]=sc.transform(a[['Time','Amount']]);return a
Xtr,Xva,Xte=tr(Xtr),tr(Xva),tr(Xte)
for n,a,b in [('train',Xtr,ytr),('val',Xva,yva),('test',Xte,yte)]:
    print(n,a.shape,'fraud',int(b.sum()))
joblib.dump((Xtr,Xva,Xte,ytr,yva,yte,sc),'data/split.joblib')
joblib.dump(list(X.columns),'models/features.joblib')


## 2. Imbalance handling experiments
Baseline vs class weights vs SMOTE vs undersampling (resampling applied to the training set only).

In [ ]:
import joblib,json,time
import numpy as np,pandas as pd
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
Xtr,Xva,Xte,ytr,yva,yte,sc=joblib.load('data/split.joblib')
spw=(ytr==0).sum()/(ytr==1).sum()
def mk(name,cw):
    if name=='LogReg': return LogisticRegression(max_iter=2000,class_weight='balanced' if cw else None)
    return XGBClassifier(n_estimators=200,max_depth=5,learning_rate=0.1,tree_method='hist',n_jobs=1,eval_metric='aucpr',scale_pos_weight=spw if cw else 1,random_state=42)
res=[]
for name in ['LogReg','XGBoost']:
    for strat in ['Baseline','Class weights','SMOTE','Undersampling']:
        t=time.time();X,y=Xtr,ytr
        if strat=='SMOTE': X,y=SMOTE(random_state=42).fit_resample(Xtr,ytr)
        if strat=='Undersampling': X,y=RandomUnderSampler(random_state=42).fit_resample(Xtr,ytr)
        m=mk(name,strat=='Class weights').fit(X,y);p=m.predict_proba(Xva)[:,1]
        thr=best_thr(yva,p);r=metrics(yva,p,thr);r.update(model=name,strategy=strat,secs=round(time.time()-t,1))
        res.append(r);print(name,strat,{k:round(v,3) if isinstance(v,float) else v for k,v in r.items() if k in('precision','recall','f1','pr_auc','secs')},flush=True)
pd.DataFrame(res).to_csv('results/imbalance_val.csv',index=False)


## 3. Model training, tuning and test evaluation
Logistic Regression, Random Forest, XGBoost and a Keras neural network. Hyperparameters chosen on validation PR-AUC; decision threshold chosen on validation F1; final numbers on the untouched test set.

In [ ]:
import joblib,json,time,itertools,os
os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
import numpy as np,pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
Xtr,Xva,Xte,ytr,yva,yte,sc=joblib.load('data/split.joblib')
spw=(ytr==0).sum()/(ytr==1).sum()
rows=[];probs={};best={}
def run(name,cands,build,fit=lambda m:m.fit(Xtr,ytr),pred=lambda m,X:m.predict_proba(X)[:,1]):
    t=time.time();bs=None
    for c in cands:
        m=fit(build(c)) ;p=pred(m,Xva);s=average_precision_score(yva,p)
        print(name,c,'val PR-AUC',round(s,4),flush=True)
        if bs is None or s>bs[0]:bs=(s,c,m)
    s,c,m=bs;pv=pred(m,Xva);thr=best_thr(yva,pv);pt=pred(m,Xte)
    r=metrics(yte,pt,thr);r.update(model=name,params=str(c),val_pr_auc=s,secs=round(time.time()-t,1));rows.append(r);probs[name]=pt;best[name]=(m,thr)
    print('>>',name,{k:(round(v,3) if isinstance(v,float) else v) for k,v in r.items()},flush=True)
run('Logistic Regression',[dict(C=c) for c in (0.01,0.1,1,10)],lambda c:LogisticRegression(max_iter=3000,class_weight='balanced',**c))
run('Random Forest',[dict(n_estimators=150,max_depth=d,min_samples_leaf=l) for d,l in ((10,2),(16,1))],
    lambda c:RandomForestClassifier(class_weight='balanced_subsample',n_jobs=1,random_state=42,max_features='sqrt',**c))
run('XGBoost',[dict(max_depth=d,learning_rate=lr,n_estimators=n,subsample=0.8,colsample_bytree=0.8) for d,lr,n in ((4,0.1,300),(6,0.05,400),(5,0.1,300))],
    lambda c:XGBClassifier(tree_method='hist',n_jobs=1,eval_metric='aucpr',random_state=42,**c))
# neural network
import tensorflow as tf
from tensorflow import keras
def build_nn(c):
    tf.random.set_seed(42);m=keras.Sequential([keras.layers.Input(shape=(30,)),keras.layers.Dense(c['u'],activation='relu'),keras.layers.Dropout(c['d']),keras.layers.Dense(c['u']//2,activation='relu'),keras.layers.Dropout(c['d']),keras.layers.Dense(1,activation='sigmoid')])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss='binary_crossentropy');return m
def fit_nn(m):
    m.fit(Xtr.values,ytr.values,validation_data=(Xva.values,yva.values),epochs=15,batch_size=2048,class_weight={0:1.0,1:float(np.sqrt(spw))},
          callbacks=[keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True,monitor='val_loss')],verbose=0);return m
run('Neural Network',[dict(u=64,d=0.2),dict(u=32,d=0.3)],build_nn,fit_nn,lambda m,X:m.predict(X.values,verbose=0).ravel())
df=pd.DataFrame(rows);df.to_csv('results/model_comparison_test.csv',index=False)
joblib.dump(probs,'results/test_probs.joblib')
for n,(m,thr) in best.items():
    if n=='Neural Network': m.save('models/nn.keras')
    else: joblib.dump(m,'models/'+n.replace(' ','_').lower()+'.joblib')
json.dump({n:b[1] for n,b in best.items()},open('models/thresholds.json','w'))
print(df[['model','precision','recall','f1','pr_auc','roc_auc','tp','fp','fn']].round(3).to_string())


## 4. Explainability (SHAP), figures and error analysis

In [ ]:
import joblib,json,numpy as np,pandas as pd,matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt,shap
from sklearn.metrics import precision_recall_curve,confusion_matrix
Xtr,Xva,Xte,ytr,yva,yte,sc=joblib.load('data/split.joblib')
probs=joblib.load('results/test_probs.joblib');thr=json.load(open('models/thresholds.json'))
cmp=pd.read_csv('results/model_comparison_test.csv');imb=pd.read_csv('results/imbalance_val.csv')
# 1 PR curves
plt.figure(figsize=(6,4.5))
for n,p in probs.items():
    pr,rc,_=precision_recall_curve(yte,p);ap=cmp.set_index('model').loc[n,'pr_auc'];plt.plot(rc,pr,label=f'{n} (PR-AUC {ap:.2f})')
plt.xlabel('Recall');plt.ylabel('Precision');plt.title('Precision-Recall curves (test set)');plt.legend(fontsize=8);plt.grid(alpha=.3);plt.tight_layout();plt.savefig('figures/pr_curves.png',dpi=160);plt.close()
# 2 confusion matrix xgb
p=probs['XGBoost'];yh=(p>=thr['XGBoost']).astype(int);cm=confusion_matrix(yte,yh)
fig,ax=plt.subplots(figsize=(4.2,3.8));ax.imshow(cm,cmap='Blues')
for i in range(2):
    for j in range(2):ax.text(j,i,f'{cm[i,j]:,}',ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=13)
ax.set_xticks([0,1],['Pred legit','Pred fraud']);ax.set_yticks([0,1],['Actual legit','Actual fraud']);ax.set_title('XGBoost confusion matrix (test)');plt.tight_layout();plt.savefig('figures/confusion_xgboost.png',dpi=160);plt.close()
# 3 imbalance chart
pv=imb.pivot(index='strategy',columns='model',values='pr_auc').loc[['Baseline','Class weights','SMOTE','Undersampling']]
pv.plot(kind='bar',figsize=(6,4));plt.ylabel('Validation PR-AUC');plt.title('Imbalance handling strategies');plt.xticks(rotation=0);plt.grid(axis='y',alpha=.3);plt.tight_layout();plt.savefig('figures/imbalance_strategies.png',dpi=160);plt.close()
# 4 model comparison
c=cmp.set_index('model')[['precision','recall','f1','pr_auc']];c.plot(kind='bar',figsize=(7,4));plt.xticks(rotation=15);plt.title('Model comparison (test set)');plt.grid(axis='y',alpha=.3);plt.legend(fontsize=8);plt.tight_layout();plt.savefig('figures/model_comparison.png',dpi=160);plt.close()
# 5 SHAP
m=joblib.load('models/xgboost.joblib');rng=np.random.RandomState(0)
idx=np.concatenate([np.where(yte.values==1)[0],rng.choice(np.where(yte.values==0)[0],1500,replace=False)])
S=Xte.iloc[idx];sv=shap.TreeExplainer(m).shap_values(S)
plt.figure();shap.summary_plot(sv,S,show=False,max_display=12);plt.tight_layout();plt.savefig('figures/shap_summary.png',dpi=160,bbox_inches='tight');plt.close()
plt.figure();shap.summary_plot(sv,S,plot_type='bar',show=False,max_display=12);plt.tight_layout();plt.savefig('figures/shap_bar.png',dpi=160,bbox_inches='tight');plt.close()
imp=pd.Series(np.abs(sv).mean(0),index=S.columns).sort_values(ascending=False);imp.head(10).to_csv('results/shap_top10.csv')
print('SHAP top10');print(imp.head(10).round(3))
# error analysis
amt_raw=sc.inverse_transform(Xte[['Time','Amount']])[:,1]
df=pd.DataFrame({'y':yte.values,'p':p,'pred':yh,'amount':amt_raw})
fn=df[(df.y==1)&(df.pred==0)];tp=df[(df.y==1)&(df.pred==1)];fp=df[(df.y==0)&(df.pred==1)]
print('FN',len(fn),'mean prob',fn.p.mean().round(3),'median amount',fn.amount.median().round(2),'| TP median amount',tp.amount.median().round(2),'| FP',len(fp),'median amount',fp.amount.median().round(2))
print('FN probs',sorted(fn.p.round(3).tolist()))
print('Total fraud amount test',round(df[df.y==1].amount.sum(),2),'caught',round(tp.amount.sum(),2),'missed',round(fn.amount.sum(),2))


In [ ]:
import joblib
sc=joblib.load('data/split.joblib')[-1]; joblib.dump(sc,'models/scaler.joblib')